In [78]:
# based on https://til.simonwillison.net/llms/python-react-pattern

In [79]:
#import openai
import re
#import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI

In [80]:
MODEL_NAME = os.getenv("OPENAI_MODEL", "qwen3-coder-next:cloud")

client = OpenAI()

In [81]:
chat_completion = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Hello world"}]
)

In [82]:
chat_completion.choices[0].message.content

"Hello! 😊  \nIt's great to meet you—how can I help today? 🌟"

In [83]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=0,
            messages=self.messages,
        )
        return completion.choices[0].message.content

In [84]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [85]:
DOG_AVERAGE_WEIGHTS = {
    "scottish terrier": 20,
    "border collie": 37,
    "toy poodle": 7,
}


def calculate(what):
    return eval(what)


def average_dog_weight(name):
    breed = name.strip().lower()
    if breed in DOG_AVERAGE_WEIGHTS:
        return f"A {name.strip()} averages {DOG_AVERAGE_WEIGHTS[breed]} lbs"
    return "An average dog weighs 50 lbs"


known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight,
}

In [86]:
abot = Agent(prompt)

In [87]:
result = abot("How much does a toy poodle weigh?")
print(result)

Thought: I should look up the average weight of a toy poodle using the average_dog_weight action.
Action: average_dog_weight: Toy Poodle
PAUSE


In [88]:
result = average_dog_weight("Toy Poodle")

In [89]:
result

'A Toy Poodle averages 7 lbs'

In [90]:
next_prompt = "Observation: {}".format(result)

In [91]:
abot(next_prompt)

'Answer: A Toy Poodle averages 7 lbs.'

In [104]:
abot.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user',
  'content': 'I have 2 dogs,

In [92]:
abot = Agent(prompt)

In [93]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)

'Thought: I need to find the average weight of a Border Collie and a Scottish Terrier using average_dog_weight, then sum them.\nAction: average_dog_weight: Border Collie\nPAUSE'

In [94]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

Observation: A Border Collie averages 37 lbs


In [95]:
abot(next_prompt)

'Thought: Now I need to find the average weight of a Scottish Terrier.\nAction: average_dog_weight: Scottish Terrier\nPAUSE'

In [96]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

Observation: A Scottish Terrier averages 20 lbs


In [97]:
abot(next_prompt)

'Thought: I have the average weights: Border Collie = 37 lbs, Scottish Terrier = 20 lbs. Now I will calculate their combined weight.\nAction: calculate: 37 + 20\nPAUSE'

In [98]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

Observation: 57


In [99]:
abot(next_prompt)

'Answer: The combined weight of the Border Collie and Scottish Terrier is 57 lbs.'

In [100]:
#Add loop

In [101]:
action_re = re.compile(r"^Action:\s+(\w+):\s+(.*)$")

In [102]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

In [103]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)

Thought: I need to find the average weight of a border collie and a scottish terrier, then sum them up.
Action: average_dog_weight: border collie
PAUSE
 -- running average_dog_weight border collie
Observation: A border collie averages 37 lbs
Thought: Now I need to find the average weight of a scottish terrier.
Action: average_dog_weight: scottish terrier
PAUSE
 -- running average_dog_weight scottish terrier
Observation: A scottish terrier averages 20 lbs
Thought: I now have the average weights for both dogs. I will sum them up to get the combined weight.
Action: calculate: 37 + 20
PAUSE
 -- running calculate 37 + 20
Observation: 57
Answer: The combined weight of your border collie and scottish terrier is 57 lbs.
